# Fact Data Processing Incremental Load

### Imports

In [0]:
from pyspark.sql.functions import *

### Schema setup for Bronze, Silver and Gold layers

In [0]:
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'

### Defining widgets for runtime configuration
#### Syntax: dbutils.widgets.text(internal handle, pre-filled value, display text)
#### /*.csv = 'read every csv file in this folder'

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'orders', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(catalog)
print(data_source)

base_path = f's3://databricks-data-engineering-project/{data_source}'
landing_path = f'{base_path}/landing/'
processed_path = f'{base_path}/processed/'

print(base_path)
print(landing_path)
print(processed_path)

fmcg
orders
s3://databricks-data-engineering-project/orders
s3://databricks-data-engineering-project/orders/landing/
s3://databricks-data-engineering-project/orders/processed/


### Read CSV from source

In [0]:
df = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(f'{landing_path}/*')
    .withColumn('read_timestamp', current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)

print(f'Row Count: {df.count()}')
display(df.limit(10))

Row Count: 8834


order_id,order_placement_date,customer_id,product_id,order_qty,read_timestamp,file_name,file_size
FDEC818102602,"Tuesday, December 16, 2025",789102,25891502,196.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",789102,25891503,null,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",789102,25891602,147.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",789102,25891101,337.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",INVALID,25891202,202.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",789102,77777777,37.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",INVALID,25891202,202.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818102602,"Tuesday, December 16, 2025",789102,25891602,147.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818501501,16/12/2025,789501,25891203,406.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060
FDEC818501501,"Tuesday, December 16, 2025",789501,25891102,463.0,2026-10-23T13:01:44.450Z,orders_2025_12_16.csv,23060


### Inspect the inferred schema

In [0]:
df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- order_placement_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- order_qty: double (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



### Ingesting raw data into Bronze layer

In [0]:
(
    df.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('append')
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')
)

### Write Incremental Batch to Bronze Staging

In [0]:
(
    df.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{bronze_schema}.incremental_staging_{data_source}')
)

### Archive Landing Files to Processed

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(file_info.path, f'{processed_path}/{file_info.name}', True)

### Silver Processing

In [0]:
df_bronze = (
    spark
    .sql(f'SELECT * FROM {catalog}.{bronze_schema}.incremental_staging_{data_source}')
)
df_bronze.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FDEC85903601|Thursday, Decembe...|     789903|  77777777|    342.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe...|     789903|  25891302|     53.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe...|     789903|  77777777|    470.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe...|     789903|  25891601|    184.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe...|     789903|  25891101|    459.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe.

### Drop Rows with Null `order_qty`

In [0]:
df_silver = df_bronze.filter(col('order_qty').isNotNull())

### Sanitize `customer_id` column

In [0]:
df_silver = (
    df_silver.withColumn('customer_id',
    when(col('customer_id').rlike(r'^[0-9]+$'), col('customer_id').cast('string'))
    .otherwise(lit(999999).cast('string')))
)

In [0]:
df_silver.show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FDEC85903601|Thursday, Decembe...|     789903|  77777777|    342.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe...|     789903|  25891302|     53.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe...|     789903|  77777777|    470.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe...|     789903|  25891601|    184.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe...|     789903|  25891101|    459.0|2026-10-23 13:01:...|orders_2025_12_04...|    22385|
|FDEC85903601|Thursday, Decembe.

### Sanitize `order_placement_date` values

In [0]:
df_silver = (
    df_silver.withColumn('order_placement_date',
    regexp_replace(col('order_placement_date'), r'^[A-Za-z]+,\s*', ''))
)

### Parse `order_placement_date` from multiple possible formats

In [0]:
df_silver = (
    df_silver.withColumn('order_placement_date',
    coalesce(
    try_to_date(col('order_placement_date'), 'yyyy/MM/dd'),
    try_to_date(col('order_placement_date'), 'dd/MM/yyyy'),
    try_to_date(col('order_placement_date'), 'yyyy-MM-dd'),
    try_to_date(col('order_placement_date'), 'dd-MM-yyyy'),
    try_to_date(col('order_placement_date'), 'MMMM dd, yyyy')
    ))
)

### Drop duplicates

In [0]:
df_silver = (
    df_silver.drop_duplicates(subset = 
    ['order_id', 'order_placement_date', 'customer_id', 'product_id', 'order_qty'])
)

### Cast `product_id` to string

In [0]:
df_silver = (
    df_silver.withColumn('product_id', col('product_id').cast('string'))
)

### Sanity Check: Earliest and Latest Dates

In [0]:
df_silver.select(
    min(col('order_placement_date')).alias('min_date'),
    max(col('order_placement_date')).alias('max_date')
).show()

+----------+----------+
|  min_date|  max_date|
+----------+----------+
|2025-12-04|2025-12-30|
+----------+----------+



#### Enrich the silver dataset by performing an inner join with the `products` table to retrieve the `product_code` for each `product_id`

In [0]:
df_products = spark.table(f'{catalog}.{silver_schema}.products')
df_joined = df_silver.join(df_products.select('product_id', 'product_code'), on = 'product_id', how = 'inner')

display(df_joined.limit(10))

product_id,order_id,order_placement_date,customer_id,order_qty,read_timestamp,file_name,file_size,product_code
25891601,FDEC86601601,2025-12-04,789601,119.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742
25891203,FDEC87501603,2025-12-04,789501,154.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2
25891601,FDEC86103601,2025-12-04,789103,130.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742
25891603,FDEC87622603,2025-12-04,789622,51.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa
25891102,FDEC86403303,2025-12-04,789403,418.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
25891403,FDEC87522503,2025-12-04,789522,342.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9
25891603,FDEC85403603,2025-12-04,789403,50.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,451f7167b28a25bde73995910e31c07dfa26411f1db47847f19e16747effbdaa
25891401,FDEC86421503,2025-12-04,789421,320.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,da6bfc596c1360ca07bda4e0ae6bfe3b8456517fc6e8ddc265630ff940f9ab05
25891503,FDEC86421503,2025-12-04,789421,225.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379
25891203,FDEC85201203,2025-12-04,789201,388.0,2026-10-23T13:01:50.699Z,orders_2025_12_04.csv,22385,889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2


### Stage `df_joined` as TempView for Silver Upsert

In [0]:
df_joined.createOrReplaceTempView('silver_orders_upsert_source')

### Merge TempView with Silver `orders` Table

In [0]:
display(spark.sql(f"""
MERGE INTO {catalog}.{silver_schema}.{data_source} AS target
USING silver_orders_upsert_source AS source
ON
    target.order_id = source.order_id
AND
    target.order_placement_date = source.order_placement_date
AND
    target.customer_id = source.customer_id
AND
    target.product_code = source.product_code

WHEN MATCHED THEN
UPDATE
    SET
        product_id = source.product_id,
        order_qty = source.order_qty
--  Note: read_timestamp, file_name and file_size were removed to maintain the original ingestion audit trail

WHEN NOT MATCHED THEN
INSERT (
    product_id,
    order_id,
    order_placement_date,
    customer_id,
    order_qty,
    read_timestamp,
    file_name,
    file_size,
    product_code
)
VALUES(
    source.product_id,
    source.order_id,
    source.order_placement_date,
    source.customer_id,
    source.order_qty,
    source.read_timestamp,
    source.file_name,
    source.file_size,
    source.product_code
);
"""))


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6947,0,0,6947


### Write Incremental Batch to Silver Staging

In [0]:
(
    df_joined.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{silver_schema}.incremental_staging_{data_source}')
)

### Gold Processing - Select final columns

In [0]:
df_gold = spark.sql(f'SELECT order_id, order_placement_date, customer_id, product_code, product_id, order_qty FROM {catalog}.{silver_schema}.incremental_staging_{data_source};')


### Stage `df_gold` as TempView for Gold Upsert

In [0]:
df_gold.createOrReplaceTempView('gold_orders_upsert_source')

### Merge TempView with Gold `staging_fact_orders` table

In [0]:
display(spark.sql(f"""
MERGE INTO {catalog}.{gold_schema}.staging_fact_{data_source} AS target
USING gold_orders_upsert_source AS source
ON
    target.order_id = source.order_id
AND
    target.order_placement_date = source.order_placement_date
AND
    target.customer_id = source.customer_id
AND
    target.product_code = source.product_code

WHEN MATCHED THEN
UPDATE
    SET
        product_id = source.product_id,
        order_qty = source.order_qty

WHEN NOT MATCHED THEN
INSERT (
    product_id,
    order_id,
    order_placement_date,
    customer_id,
    order_qty,
    product_code
)
VALUES(
    source.product_id,
    source.order_id,
    source.order_placement_date,
    source.customer_id,
    source.order_qty,
    source.product_code
);
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
6947,0,0,6947


### Extract Distinct `month_start` Dates

In [0]:
df_incremental = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.incremental_staging_{data_source};')

df_affected_months = (
    df_incremental
    .select(trunc('order_placement_date', 'MM').alias('month_start'))
    .distinct()
)

display(df_affected_months)

df_affected_months.createOrReplaceTempView('affected_months')

month_start
2025-12-01


### Re-aggregate `affected_months` (Combine Historical and Incremental Data)

**How it works:**
1. **JOIN:** Matches the staging data only to the months present in `affected_months`.
2. **GROUP BY:** Groups the new daily records into the monthly grain (month, product, customer).
3. **SELECT:** Sums the `order_qty` renames the columns for Gold MERGE.

In [0]:
df_monthly = (
    spark.sql(f"""
    SELECT
        trunc(sfo.order_placement_date, 'MM') AS date,
        sfo.product_code,
        sfo.customer_id AS customer_code,
        SUM(sfo.order_qty) AS sold_quantity
    FROM 
        {catalog}.{gold_schema}.staging_fact_{data_source} AS sfo
    INNER JOIN
        affected_months AS am
    ON
        trunc(sfo.order_placement_date, 'MM') = am.month_start
    GROUP BY
        trunc(sfo.order_placement_date, 'MM'),
        sfo.product_code,
        sfo.customer_id;
""")
)

print(f'Row Count: {df_monthly.count()}')
df_monthly.show(10)

Row Count: 612
+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-12-01|c68834ceaff15846b...|       789201|        561.0|
|2025-12-01|e91ba9d665f90254d...|       789402|       5125.0|
|2025-12-01|77b6f538a9d0e0cf8...|       999999|      18053.0|
|2025-12-01|889c67757ece9c973...|       789321|       2782.0|
|2025-12-01|778c2a7aa27bfdb21...|       789303|       1409.0|
|2025-12-01|e92c739a8d78cd6cb...|       789420|       4475.0|
|2025-12-01|e91ba9d665f90254d...|       789301|       3973.0|
|2025-12-01|0cb7b2f42657b625f...|       999999|      12283.0|
|2025-12-01|062f5574bbdf4386b...|       789702|       1415.0|
|2025-12-01|102628255d24304d6...|       789422|       6006.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


### Stage `df_monthly` as TempView for MERGE

In [0]:
df_monthly.createOrReplaceTempView('monthly_src')

### Merge TempView with parent

In [0]:
display(spark.sql(f"""
MERGE INTO {catalog}.{gold_schema}.fact_{data_source} AS target
USING monthly_src AS source
ON
    target.date = source.date
AND
    target.product_code = source.product_code
AND
    target.customer_code = source.customer_code
-- match on full monthly grain (date + product + customer) to prevent multi-month collisions

WHEN MATCHED THEN
UPDATE
    SET
        sold_quantity = source.sold_quantity

WHEN NOT MATCHED THEN
INSERT (
    date,
    product_code,
    customer_code,
    sold_quantity
)
VALUES (
    source.date,
    source.product_code,
    source.customer_code,
    source.sold_quantity
);
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
612,462,0,150


### Cleanup Staging Tables

In [0]:
%sql

DROP TABLE IF EXISTS fmcg.bronze.incremental_staging_orders;
DROP TABLE IF EXISTS fmcg.silver.incremental_staging_orders;